In [1]:
import rasterio
import numpy as np
from rasterio.merge import merge
import os

In [2]:


file_path = "../output/fourier/NKD/"
# Danh sách các file TIF
file_list = [
     "../output/fourier/NKD/lt_pred_cut.tif",
    "../output/fourier/NKD/cln_pred_cut.tif",
    "../output/fourier/NKD/ctxd_pred_cut.tif",
    "../output/fourier/NKD/lua2v_pred_cut.tif",
    "../output/fourier/NKD/lua3v_pred_cut.tif",
    "../output/fourier/NKD/song_pred_cut.tif",
    "../output/fourier/NKD/ts_pred_cut.tif"
]

# full_file_paths = [os.path.join(file_path, file) for file in file_list]

In [3]:
datasets = [rasterio.open(file) for file in file_list]
data_arrays = [ds.read(1) for ds in datasets]  # Đọc band đầu tiên của mỗi file


In [4]:
# Kiểm tra số band của mỗi file
for i, ds in enumerate(datasets):
    print(f"File {file_list[i]}: {ds.count} bands")

File ../output/fourier/NKD/lt_pred_cut.tif: 13 bands
File ../output/fourier/NKD/cln_pred_cut.tif: 13 bands
File ../output/fourier/NKD/ctxd_pred_cut.tif: 13 bands
File ../output/fourier/NKD/lua2v_pred_cut.tif: 13 bands
File ../output/fourier/NKD/lua3v_pred_cut.tif: 13 bands
File ../output/fourier/NKD/song_pred_cut.tif: 13 bands
File ../output/fourier/NKD/ts_pred_cut.tif: 13 bands


In [5]:
# Ghép dữ liệu: với mỗi band, lấy giá trị max từ 6 file
combined_bands = []
num_bands = 13  # Số band mỗi file
for band_idx in range(1, num_bands + 1):
    # Đọc band hiện tại từ tất cả 6 file
    band_data = [ds.read(band_idx) for ds in datasets]
    # Chuyển thành mảng 3D (6 file, height, width)
    band_stack = np.stack(band_data, axis=0)
    # Tính giá trị max theo trục file (trục 0)
    band_max = np.max(band_stack, axis=0)
    combined_bands.append(band_max)

In [6]:
# Chuyển danh sách thành mảng 3D (13 band, height, width)
combined_bands = np.stack(combined_bands, axis=0)

In [7]:
meta = datasets[0].meta.copy()
meta.update({
    "count": num_bands,  # File đầu ra có 13 band
    "height": combined_bands.shape[1],
    "width": combined_bands.shape[2]
})

In [8]:
with rasterio.open("../output/fourier/NKD/max_combined_13bands.tif", "w", **meta) as dest:
    for band_idx in range(num_bands):
        dest.write(combined_bands[band_idx], band_idx + 1)

# Đóng file
for ds in datasets:
    ds.close()